In [0]:
#Helper function to add the file metadata for ingestion (source file and ingestion_timestamp)

from pyspark.sql import functions as f

def add_ingestion_metadata(df):
    return (
        df \
        .withColumn('current_timestamp',f.current_timestamp())
        .withColumn('source_file',f.col('_metadata.file_path'))
    )

In [0]:
def write_to_bronze(
    input_df,
    target_table,
    batch_id
):
    final_df = input_df.withColumn('batch_id',f.lit(batch_id))
    (
        final_df
            .write
            .format('delta')
            .mode('overwrite')
            .partitionBy('batch_id')
            .option('replaceWhere',f"batch_id='{batch_id}'")
            .saveAsTable(target_table)
    )